In [ ]:
from pathlib import Path

import pandas as pd
from google.colab import drive

drive.mount("/content/drive")

data_dir = Path("/content/drive/MyDrive/exsu_project/prepared_data")

train = pd.read_csv(data_dir / "PubMed_20k_RCT_train.csv", keep_default_na=False)
dev = pd.read_csv(data_dir / "PubMed_20k_RCT_dev.csv", keep_default_na=False)

assert len(train) == 180_040, "Unexpected number of training sentences"
assert len(dev) == 30_212, "Unexpected number of validation sentences"

print(f"Train: {len(train):,} sentences")
print(f"Dev: {len(dev):,} sentences")

Mounted at /content/drive
Train: 180,040 sentences
Dev: 30,212 sentences


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import make_pipeline

model = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=50_000),
    LogisticRegression(C=1.0, max_iter=500),
)

print("Training Logistic Regression...")
model.fit(train["text"], train["label"])

dev_predictions = model.predict(dev["text"])

print(f"Dev accuracy: {accuracy_score(dev['label'], dev_predictions):.4f}")
print(f"Dev macro-F1: {f1_score(dev['label'], dev_predictions, average='macro'):.4f}")

Training Logistic Regression...
Dev accuracy: 0.8310
Dev macro-F1: 0.7669


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

labels = ["BACKGROUND", "OBJECTIVE", "METHODS", "RESULTS", "CONCLUSIONS"]

print(classification_report(
    dev["label"], dev_predictions, labels=labels, digits=4, zero_division=0
))

counts = confusion_matrix(dev["label"], dev_predictions, labels=labels)
display(pd.DataFrame(
    counts,
    index=pd.Index(labels, name="Actual"),
    columns=pd.Index(labels, name="Predicted"),
))

              precision    recall  f1-score   support

  BACKGROUND     0.6494    0.6712    0.6601      3449
   OBJECTIVE     0.7258    0.5337    0.6151      2376
     METHODS     0.8788    0.9287    0.9031      9964
     RESULTS     0.8924    0.8941    0.8933      9841
 CONCLUSIONS     0.7692    0.7571    0.7631      4582

    accuracy                         0.8310     30212
   macro avg     0.7831    0.7570    0.7669     30212
weighted avg     0.8284    0.8310    0.8283     30212



Predicted,BACKGROUND,OBJECTIVE,METHODS,RESULTS,CONCLUSIONS
Actual,,,,,
BACKGROUND,2315,365,275,36,458
OBJECTIVE,738,1268,172,23,175
METHODS,117,68,9254,452,73
RESULTS,26,2,679,8799,335
CONCLUSIONS,369,44,150,550,3469


In [ ]:
predictions_dir = data_dir.parent / "predictions"
predictions_dir.mkdir(parents=True, exist_ok=True)

output = dev[["abstract_id", "sentence_idx", "label"]].copy()
output["prediction"] = dev_predictions

probabilities = pd.DataFrame(
    model.predict_proba(dev["text"]),
    columns=model.classes_,
    index=dev.index,
)

for label in labels:
    output[f"prob_{label}"] = probabilities[label]

path = predictions_dir / "dev_logistic_regression_text.csv"
output.to_csv(path, index=False)
print(f"Saved {len(output):,} predictions to {path}")

Saved 30,212 predictions to /content/drive/MyDrive/exsu_project/predictions/dev_logistic_regression_text.csv


In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score

threshold = 0.50
rows = []

for label in labels:
    actual = output["label"].eq(label)
    scores = output[f"prob_{label}"]
    predicted = scores.ge(threshold)

    rows.append({
        "Class": label,
        "AUROC": roc_auc_score(actual, scores),
        "AUPRC (AP)": average_precision_score(actual, scores),
        "Sensitivity": predicted[actual].mean(),
        "Specificity": (~predicted[~actual]).mean(),
    })

metrics = pd.DataFrame(rows).set_index("Class")
metrics.loc["Macro average"] = metrics.mean()

print(f"Development set | One class vs. the other four | Threshold: {threshold:.2f}")
display(metrics.round(4))

Development set | One class vs. the other four | Threshold: 0.50


,AUROC,AUPRC (AP),Sensitivity,Specificity
Class,,,,
BACKGROUND,0.9507,0.7041,0.5436,0.9707
OBJECTIVE,0.9536,0.7004,0.4508,0.9890
METHODS,0.9773,0.9501,0.9012,0.9573
RESULTS,0.9731,0.9520,0.8684,0.9593
CONCLUSIONS,0.9599,0.8430,0.6656,0.9756
Macro average,0.9629,0.8299,0.6859,0.9704


In [ ]:
import numpy as np

# Count actual and predicted label pairs separately for each abstract.
abstract_index, abstract_ids = pd.factorize(output["abstract_id"])
actual = pd.Categorical(output["label"], categories=labels).codes
predicted = pd.Categorical(output["prediction"], categories=labels).codes

n_abstracts = len(abstract_ids)
n_classes = len(labels)
counts_by_abstract = np.zeros((n_abstracts, n_classes, n_classes), dtype=int)
np.add.at(counts_by_abstract, (abstract_index, actual, predicted), 1)

rng = np.random.default_rng(42)
bootstrap_scores = np.empty(2_000)

for i in range(len(bootstrap_scores)):
    sample = rng.integers(0, n_abstracts, size=n_abstracts)
    cm = counts_by_abstract[sample].sum(axis=0)
    denominator = cm.sum(axis=0) + cm.sum(axis=1)
    class_f1 = np.divide(
        2 * np.diag(cm),
        denominator,
        out=np.zeros(n_classes),
        where=denominator > 0,
    )
    bootstrap_scores[i] = class_f1.mean()

point_estimate = f1_score(
    output["label"], output["prediction"], labels=labels, average="macro"
)
lower, upper = np.percentile(bootstrap_scores, [2.5, 97.5])

print(f"Dev abstracts: {n_abstracts:,}")
print(f"Dev macro-F1: {point_estimate:.4f}")
print(f"95% bootstrap CI: {lower:.4f} to {upper:.4f}")

Dev abstracts: 2,500
Dev macro-F1: 0.7669
95% bootstrap CI: 0.7599 to 0.7737


In [ ]:
import joblib

model_dir = data_dir.parent / "models"
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / "logistic_regression_text.joblib"
joblib.dump(model, model_path)

print(f"Saved model to {model_path}")

Saved model to /content/drive/MyDrive/exsu_project/models/logistic_regression_text.joblib


In [1]:
results_dir = data_dir.parent / "results"
results_dir.mkdir(parents=True, exist_ok=True)

summary = pd.DataFrame([{
    "model": "TF-IDF Logistic Regression (text only)",
    "split": "dev",
    "sentences": len(dev),
    "abstracts": n_abstracts,
    "accuracy": accuracy_score(dev["label"], dev_predictions),
    "macro_f1": point_estimate,
    "macro_f1_ci95_lower": lower,
    "macro_f1_ci95_upper": upper,
    "macro_auroc": metrics.loc["Macro average", "AUROC"],
    "macro_average_precision": metrics.loc["Macro average", "AUPRC (AP)"],
}])

summary.to_csv(results_dir / "dev_logistic_regression_summary.csv", index=False)
metrics.assign(threshold=threshold).to_csv(
    results_dir / "dev_logistic_regression_per_class.csv"
)

pd.DataFrame(
    counts,
    index=pd.Index(labels, name="Actual"),
    columns=pd.Index(labels, name="Predicted"),
).to_csv(results_dir / "dev_logistic_regression_confusion_matrix.csv")

print(f"Saved 3 result tables to {results_dir}")

NameError: name 'data_dir' is not defined

In [2]:
from pathlib import Path
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")

project_dir = Path("/content/drive/MyDrive/exsu_project")
predictions = pd.read_csv(
    project_dir / "predictions" / "dev_logistic_regression_text.csv",
    keep_default_na=False,
)

labels = ["BACKGROUND", "OBJECTIVE", "METHODS", "RESULTS", "CONCLUSIONS"]
rows = []

for label in labels:
    actual = predictions["label"].eq(label)
    predicted = predictions["prediction"].eq(label)

    rows.append({
        "Class": label,
        "Sensitivity": predicted[actual].mean(),
        "Specificity": (~predicted[~actual]).mean(),
    })

table = pd.DataFrame(rows).set_index("Class")
table.loc["Macro average"] = table.mean()

results_dir = project_dir / "results"
results_dir.mkdir(parents=True, exist_ok=True)
table.to_csv(results_dir / "dev_logistic_regression_sensitivity_specificity_argmax.csv")

print("Development set | Decision rule: highest-probability class")
display(table.round(4))

Mounted at /content/drive
Development set | Decision rule: highest-probability class


,Sensitivity,Specificity
Class,,
BACKGROUND,0.6712,0.9533
OBJECTIVE,0.5337,0.9828
METHODS,0.9287,0.9370
RESULTS,0.8941,0.9479
CONCLUSIONS,0.7571,0.9594
Macro average,0.7570,0.9561


For this run, the logistic regression regularization parameter C was fixed at 1.0 and was not tuned.